# cAPTure: XGB-P+T development training

This CPU notebook builds label-free window-close context for the five approved development scenarios, then trains one depth-5 XGBoost model per benign-background fold. It reuses the audited 103-column packet representation and adds the 14 frozen context fields. Every validation packet receives one OOF score. Run the two folds sequentially.

The XGB-P sanity review is recorded in the manifest and `readiness.xgb_p_sanity_gate_passed` is true. The runner verifies that gate before fitting. Do not use held-out author-train or final-test scenario contents.


## 1. Prepare the Colab environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_xgb_p_t_work")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/utils/capture_xgb_p_t.py", "code/python/tests/test_capture_xgb_p_t.py", "code/python/requirements-capture-xgb.txt"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("XGB-P+T environment is ready.")
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


## 2. Run synthetic protocol checks


In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for pattern in ("test_capture_preprocess.py", "test_capture_xgb_p.py", "test_capture_xgb_p_t.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "code/python/tests"), "-p", pattern, "-v"], env=test_environment, cwd=PROJECT_ROOT, check=True)
print("Synthetic packet and context checks passed.")


## 3. Bind completed development artifacts

Set `RUN_ID` to an existing run ID only when resuming. A new ID creates new immutable output directories. Context generation is run once per ID; both training folds reuse it.


In [ ]:
from utils.capture_data import load_manifest
from utils.capture_xgb_p_t import (
    build_context_run, run_xgb_p_t_fold, summarize_xgb_p_t_oof,
    compare_xgb_p_t_to_xgb_p,
    validate_context_run, validate_xgb_p_t_fold_run,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
RUN_ID = None  # Replace with an earlier XGB-P+T run ID only when resuming.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p_t"
CONTEXT_DIR = DRIVE_ROOT / "xgb_p_t_context_runs" / RUN_ID
TRAINING_DIR = DRIVE_ROOT / "xgb_p_t_runs" / RUN_ID
BATCH_SIZE = 50_000
NTHREAD = 2
manifest = load_manifest(MANIFEST_PATH)
print("Context output:", CONTEXT_DIR)
print("Training output:", TRAINING_DIR)
print("Sanity gate passed:", manifest["readiness"]["xgb_p_sanity_gate_passed"])


## 4. Build and audit label-free context

This step reads only the five prepared development packet tables. It writes no model scores. The checksum and row count of each context artifact are verified before reuse.


In [ ]:
if CONTEXT_DIR.exists():
    context_report = validate_context_run(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, context_dir=CONTEXT_DIR)
else:
    context_report = build_context_run(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, output_dir=CONTEXT_DIR, batch_size=BATCH_SIZE)
display(pd.DataFrame.from_dict(context_report["scenarios"], orient="index")[["rows", "nonempty_windows", "context_sha256"]])
print("Context feature count:", len(context_report["context_columns"]))


## 5. Train fold A, then fold B

The runner checks the documented XGB-P sanity gate. It fits the context scaler on the current training fold only, uses the same packet features, class weights, depth-5/200-round model settings, and window-close decision time as XGB-P, and verifies the copied artifacts.


In [ ]:
def train_or_verify(fold):
    output_dir = TRAINING_DIR / "depth5_primary" / f"fold_{fold}"
    if output_dir.exists():
        return validate_xgb_p_t_fold_run(output_dir, fold)
    return run_xgb_p_t_fold(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR, context_dir=CONTEXT_DIR, output_dir=output_dir, local_work_root=LOCAL_WORK_ROOT, fold=fold, batch_size=BATCH_SIZE, nthread=NTHREAD)

fold_a = train_or_verify("A")
display(pd.DataFrame.from_dict(fold_a["validation"], orient="index")[["rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])
print("Fold A macro packet ROC-AUC:", fold_a["fold_macro_packet_roc_auc"])


In [ ]:
fold_b = train_or_verify("B")
display(pd.DataFrame.from_dict(fold_b["validation"], orient="index")[["rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])
print("Fold B macro packet ROC-AUC:", fold_b["fold_macro_packet_roc_auc"])


## 6. Verify aligned OOF results against XGB-P

This reads the archived XGB-P and new XGB-P+T predictions, verifies their packet keys and aligned decision times, then compares the predeclared hierarchical macro ROC-AUC. Threshold, alert-budget, sequence, and latency analysis remains a separate frozen evaluation step.


In [ ]:
summary = summarize_xgb_p_t_oof(TRAINING_DIR)
comparison = compare_xgb_p_t_to_xgb_p(baseline_run_dir=BASELINE_RUN_DIR, context_run_dir=TRAINING_DIR, batch_size=BATCH_SIZE)
display(pd.DataFrame.from_dict(comparison["aligned_oof_scenarios"], orient="index"))
print("XGB-P hierarchical macro OOF packet ROC-AUC:", comparison["xgb_p_hierarchical_macro_oof_packet_roc_auc"])
print("XGB-P+T hierarchical macro OOF packet ROC-AUC:", comparison["xgb_p_t_hierarchical_macro_oof_packet_roc_auc"])
print("Hierarchical macro ROC-AUC delta:", comparison["delta_hierarchical_macro_oof_packet_roc_auc"])
print("Thresholds selected:", comparison["thresholds_selected"])
